# Supply Chain Data Analyst — тестовое задание (Генерация)

Расчёт рекомендуемого количества к заказу (`RecommendedOrder`) по 200 SKU на дату 19.09.2025.

## Настройка

Импорты и настраиваемые параметры — общие для всего ноутбука. Все новые импорты добавляются в ячейку ниже, все новые параметры — в ячейку после неё, а не в код по ходу работы.

In [1]:
# Path — для удобной и переносимой работы с файловыми путями.
from pathlib import Path

# pandas — основной инструмент для чтения Excel и работы с таблицами.
import pandas as pd

# norm — для z-квантиля нормального распределения в формуле страхового запаса.
from scipy.stats import norm

In [2]:
# Имена файлов с исходными данными — если файлы переименуют, меняем только здесь.
CANDIDATE_FILE_NAME = "candidate_data.xlsx"
DICTIONARY_FILE_NAME = "data_dictionary.xlsx"

# Дата финального расчёта RecommendedOrder — задана условием задания.
CALCULATION_DATE = pd.Timestamp("2025-09-19")

# Целевой уровень сервиса для доработанной модели (эмпирический квантиль). Не задан в задании явно -
# 95% выбран как разумная отправная точка, чувствительность к этому значению проверяем отдельно.
SERVICE_LEVEL = 0.95

## Функции

Все переиспользуемые функции проекта — в одном месте, а не разбросаны по шагам, где впервые понадобились. Каждая функция определена один раз здесь, проверяется/используется — в соответствующем шаге дальше по ноутбуку.

### `get_protection_period_days` — protection period на произвольную дату

Срок поставки (константа `LEAD_TIME_DAYS`) + интервал до следующего заказа из `OrderCalendar`. Используется и в бэктесте (раздел 2), и в финальном расчёте (раздел 3).

In [3]:
def get_protection_period_days(order_date):
    """Protection period = LeadTimeDays + интервал до следующего заказа из OrderCalendar."""
    # Находим строку календаря для этой даты заказа.
    row = order_calendar_df[order_calendar_df["OrderDate"] == order_date]
    assert len(row) == 1, f"Ожидали одну строку календаря на {order_date}, нашли {len(row)}"
    return int(row["OrderCycleDays"].iloc[0]) + LEAD_TIME_DAYS

### `get_naive_demand_estimate` — наивная оценка спроса (для baseline)

Среднесуточные продажи по каждому SKU, посчитанные только по истории строго до даты решения — без утечки будущего. Используется в baseline-модели (без поправок и без страхового запаса).

In [4]:
def get_naive_demand_estimate(as_of_date):
    """Средний дневной спрос по каждому SKU, посчитанный только по дням строго до as_of_date."""
    # Берём только "прошлое" относительно даты решения — без подглядывания в будущее.
    past_history = history_df[history_df["Date"] < as_of_date]
    return past_history.groupby("SKU")["SalesQty"].mean().rename("mean_daily_sales")

### `get_inventory_position_asof` — позиция запаса на историческую дату

Фактический остаток (с заполненными пропусками) на конец дня перед указанной датой. `InTransit`/`OpenPO` для исторических дат принимаются за 0 — данных о них нет (см. допущения бэктеста). Возвращает саму позицию и число SKU, для которых пришлось поставить fallback 0 (нет ни одного наблюдения).

In [5]:
def get_inventory_position_asof(control_date):
    """Стартовая позиция запаса = остаток (с ffill) на конец дня перед control_date. InTransit/OpenPO = 0."""
    prev_day = control_date - pd.Timedelta(days=1)
    day_slice = history_sorted[history_sorted["Date"] == prev_day]
    position = day_slice.set_index("SKU")["stock_filled"].rename("inventory_position")

    # У части SKU нет вообще ни одного наблюдения остатка до этой даты (ffill заполнить нечем) —
    # для них считаем позицию запаса неизвестной, поэтому явно нулевой, а не выдумываем число.
    n_missing = position.isna().sum()
    if n_missing:
        position = position.fillna(0.0)
    return position, n_missing

### `get_actual_demand_over_window` — реализованный спрос за period (для проверки решений)

Сумма фактических продаж по SKU за окно `[control_date, control_date + protection_period_days − 1]`. Это "истина" из факта, с которой бэктест сравнивает решение модели.

In [6]:
def get_actual_demand_over_window(control_date, protection_period_days):
    """Сумма фактических продаж по каждому SKU за [control_date, control_date + protection_period_days - 1]."""
    window_end = control_date + pd.Timedelta(days=protection_period_days - 1)
    window = history_df[(history_df["Date"] >= control_date) & (history_df["Date"] <= window_end)]
    return window.groupby("SKU")["SalesQty"].sum().rename("actual_demand")

### `get_corrected_past_history` — история с поправкой на дефицит (censored demand)

В дни с наблюдаемым нулевым остатком `SalesQty`, вероятно, занижает истинный спрос — заменяем на среднее по этому же SKU в дни с товаром в наличии. Используется в доработанной модели (не в baseline — там сырые продажи намеренно).

In [7]:
def get_corrected_past_history(as_of_date):
    """История до as_of_date с поправкой на дефицит: в дни с наблюдаемым нулевым остатком
    SalesQty заменяется на среднее по этому же SKU в дни с положительным остатком."""
    past = history_df[history_df["Date"] < as_of_date].copy()

    is_oos = past["Остаток на конец дня"] == 0
    mean_when_in_stock = past.loc[~is_oos].groupby("SKU")["SalesQty"].mean()

    # astype(float) - иначе pandas ругается при записи дробного среднего в исходную целочисленную колонку.
    past["SalesQty_corrected"] = past["SalesQty"].astype(float)
    past.loc[is_oos, "SalesQty_corrected"] = past.loc[is_oos, "SKU"].map(mean_when_in_stock)

    # Если у SKU вообще не было дней с товаром в наличии - подстановка даст NaN, оставляем как есть
    # (обычно и так 0 - это тот же случай "недостаточно данных", что и раньше).
    past["SalesQty_corrected"] = past["SalesQty_corrected"].fillna(past["SalesQty"])
    return past

### `get_S_with_safety_stock` — S для доработанной модели: дни защиты + страховой запас

`S = скорректированный средний спрос × protection period + z(Service Level) × std скользящих сумм спроса за такие периоды` (тоже на скорректированных данных). Замена сырому эмпирическому квантилю — та же логика сервиса, но без раздувания запаса почти вдвое.

In [8]:
def get_S_with_safety_stock(as_of_date, protection_period_days, service_level):
    """S = дни защиты (скорректированный средний спрос x period) + страховой запас (z x std
    скользящих сумм спроса за такие периоды, тоже на скорректированных данных)."""
    corrected = get_corrected_past_history(as_of_date).sort_values(["SKU", "Date"])

    # "Дни защиты" - та же логика, что и в baseline (шаг 15), но на скорректированном спросе.
    mean_daily = corrected.groupby("SKU")["SalesQty_corrected"].mean()
    protection_days_component = mean_daily * protection_period_days

    # Страховой запас: эмпирическое std скользящих сумм за period (не daily_std*sqrt(period) -
    # считаем изменчивость сразу на уровне periода, без допущения о независимости дней).
    rolling_sum = (
        corrected.groupby("SKU")["SalesQty_corrected"]
        .rolling(protection_period_days)
        .sum()
        .reset_index(level=0)
    )
    std_period = rolling_sum.groupby("SKU")["SalesQty_corrected"].std()

    z = norm.ppf(service_level)
    safety_stock = (z * std_period).clip(lower=0)

    S = (
        protection_days_component.reindex(sku_df["SKU"]).fillna(0.0)
        + safety_stock.reindex(sku_df["SKU"]).fillna(0.0)
    )
    return S

### `compute_baseline_decisions` — сборка решений baseline-модели по датам

Прогоняет наивную формулу (`get_naive_demand_estimate` + `get_inventory_position_asof`) по каждой дате из переданного набора, сразу считает `available_to_cover`, `shortfall`, `excess`, `cycle_success` относительно факта (`actual_demand_df`).

In [9]:
def compute_baseline_decisions(dates_df):
    """Решения baseline-модели (среднее по всей истории до даты, без поправок) на переданных датах."""
    rows = []
    for _, cal_row in dates_df.iterrows():
        control_date = cal_row["OrderDate"]
        protection_period_days = int(cal_row["protection_period_days"])

        demand_estimate = get_naive_demand_estimate(control_date)
        inventory_position, _ = get_inventory_position_asof(control_date)

        decision = pd.DataFrame({"SKU": sku_df["SKU"]})
        decision["control_date"] = control_date
        decision["protection_period_days"] = protection_period_days
        decision = decision.merge(demand_estimate, on="SKU", how="left")
        decision = decision.merge(inventory_position, on="SKU", how="left")

        decision["S"] = decision["mean_daily_sales"] * decision["protection_period_days"]
        gap = decision["S"] - decision["inventory_position"]
        decision["RecommendedOrder"] = gap.round().clip(lower=0).astype(int)
        rows.append(decision)

    decisions_df = pd.concat(rows, ignore_index=True)
    decisions_df = decisions_df.merge(actual_demand_df, on=["SKU", "control_date"], how="left")

    decisions_df["available_to_cover"] = decisions_df["inventory_position"] + decisions_df["RecommendedOrder"]
    decisions_df["shortfall"] = (decisions_df["actual_demand"] - decisions_df["available_to_cover"]).clip(lower=0)
    decisions_df["excess"] = (decisions_df["available_to_cover"] - decisions_df["actual_demand"]).clip(lower=0)
    decisions_df["cycle_success"] = decisions_df["shortfall"] == 0
    return decisions_df

### `build_decisions` — сборка решений по датам для произвольной функции расчёта `S`

Обобщённая версия `compute_baseline_decisions`: принимает саму функцию расчёта `S` параметром, поэтому один и тот же код обслуживает и доработанную модель (`get_S_with_safety_stock`), и чувствительность к Service Level (та же функция с разными `service_level`).

In [10]:
def build_decisions(dates_df, S_func, **S_kwargs):
    """Общая сборка таблицы решений SKU x даты - принимает функцию расчёта S, чтобы не дублировать код
    между baseline и доработанной моделью (отличаются только тем, как считается S)."""
    rows = []
    for _, cal_row in dates_df.iterrows():
        control_date = cal_row["OrderDate"]
        protection_period_days = int(cal_row["protection_period_days"])

        S = S_func(control_date, protection_period_days, **S_kwargs)
        inventory_position, _ = get_inventory_position_asof(control_date)

        decision = pd.DataFrame({"SKU": sku_df["SKU"]})
        decision["control_date"] = control_date
        decision["protection_period_days"] = protection_period_days
        decision = decision.merge(S.rename("S"), on="SKU", how="left")
        decision = decision.merge(inventory_position, on="SKU", how="left")

        gap = decision["S"] - decision["inventory_position"]
        decision["RecommendedOrder"] = gap.round().clip(lower=0).astype(int)
        rows.append(decision)

    decisions_df = pd.concat(rows, ignore_index=True)
    decisions_df = decisions_df.merge(actual_demand_df, on=["SKU", "control_date"], how="left")

    decisions_df["available_to_cover"] = decisions_df["inventory_position"] + decisions_df["RecommendedOrder"]
    decisions_df["shortfall"] = (decisions_df["actual_demand"] - decisions_df["available_to_cover"]).clip(lower=0)
    decisions_df["excess"] = (decisions_df["available_to_cover"] - decisions_df["actual_demand"]).clip(lower=0)
    decisions_df["cycle_success"] = decisions_df["shortfall"] == 0
    return decisions_df

### `compute_metrics` — сводные метрики по таблице решений

Cycle Service Level, Fill Rate, средний/максимальный запас, суммарный дефицит/излишек — одной строкой, из любой таблицы, собранной `compute_baseline_decisions`/`build_decisions`. Переиспользуется для baseline, доработанной модели и анализа чувствительности.

In [11]:
def compute_metrics(decisions_df, model_name):
    """Сводная строка метрик по таблице решений — переиспользуем для baseline и для доработанной модели."""
    return pd.DataFrame([{
        "model": model_name,
        "cycle_service_level": decisions_df["cycle_success"].mean(),
        "fill_rate": 1 - decisions_df["shortfall"].sum() / decisions_df["actual_demand"].sum(),
        "avg_inventory": decisions_df["available_to_cover"].mean(),
        "max_inventory": decisions_df["available_to_cover"].max(),
        "total_shortfall": decisions_df["shortfall"].sum(),
        "total_excess": decisions_df["excess"].sum(),
    }])

# Раздел 1. Baseline

## Шаг 1. Пути к данным

Определяем расположение исходных файлов (`candidate_data.xlsx`, `data_dictionary.xlsx`) относительно ноутбука и проверяем, что они на месте.

In [12]:
# Ноутбук лежит в notebooks/, поэтому поднимаемся на один уровень вверх, чтобы попасть в корень проекта.
project_dir = Path.cwd().parent

# Папка с исходными данными — на уровне корня проекта, не внутри notebooks/.
data_dir = project_dir / "data"

# Полные пути к обоим Excel-файлам, собранные из констант в ячейке параметров.
candidate_path = data_dir / CANDIDATE_FILE_NAME
dictionary_path = data_dir / DICTIONARY_FILE_NAME

# Проверяем, что оба файла реально существуют по этим путям, прежде чем пытаться их читать.
# Если assert сработает — сразу понятно, что не так, вместо непонятной ошибки на этапе чтения Excel.
assert candidate_path.exists(), f"Не найден файл: {candidate_path}"
assert dictionary_path.exists(), f"Не найден файл: {dictionary_path}"

print("candidate_data.xlsx найден:", candidate_path)
print("data_dictionary.xlsx найден:", dictionary_path)

candidate_data.xlsx найден: C:\Users\User\Desktop\clode folder\projects_code\тестовое от Генерации\reorder-model\data\candidate_data.xlsx
data_dictionary.xlsx найден: C:\Users\User\Desktop\clode folder\projects_code\тестовое от Генерации\reorder-model\data\data_dictionary.xlsx


## Шаг 2. Структура Excel-файлов

Смотрим список листов в обоих файлах, не загружая данные целиком.

In [13]:
# pd.ExcelFile открывает файл и читает его оглавление (список листов),
# но не загружает содержимое листов в память — это быстрее, чем сразу читать все данные.
candidate_excel = pd.ExcelFile(candidate_path)
dictionary_excel = pd.ExcelFile(dictionary_path)

print("Листы в candidate_data.xlsx:", candidate_excel.sheet_names)
print("Листы в data_dictionary.xlsx:", dictionary_excel.sheet_names)

Листы в candidate_data.xlsx: ['SKU', 'DailyHistory', 'OpenSupply', 'OrderCalendar']
Листы в data_dictionary.xlsx: ['Поля', 'Условия']


## Шаг 3. Словарь полей

Читаем лист "Поля" — описание всех колонок в данных, чтобы дальше корректно их интерпретировать.

In [14]:
# Читаем лист "Поля" целиком — он небольшой, это справочная таблица, а не данные для расчёта.
fields_df = pd.read_excel(dictionary_path, sheet_name="Поля")

# pd.set_option, чтобы длинные текстовые описания не обрезались "...".
pd.set_option("display.max_colwidth", None)

fields_df

,Лист,Поле,Единицы,Определение
0,SKU,SKU,Текст,Идентификатор позиции активного ассортимента. Один SKU — одна строка.
1,SKU,CurrentStock,Шт.,"Текущий остаток на дату расчёта 19.09.2025, до размещения заказа."
2,SKU,InTransit,Шт.,"Отгруженное, ещё не полученное количество. Сумма строк OpenSupply со Status = InTransit."
3,SKU,OpenPO,Шт.,"Размещённое, ещё не отгруженное количество. Сумма строк OpenSupply со Status = OpenPO."
4,SKU,LeadTimeDays,Календарные дни,Учебный срок нового заказа — 10 дней. Для уже размещённых поставок используйте ExpectedReceiptDate.
5,DailyHistory,Date,Дата,День наблюдения. Одна строка на SKU и календарный день.
6,DailyHistory,SKU,Текст,"Идентификатор, связанный с листом SKU."
7,DailyHistory,SalesQty,Шт.,Фактически выполненные продажи за указанный день. Возвраты не вычитаются.
8,DailyHistory,Остаток на конец дня,Шт.,Остаток на конец указанного дня. Пустая ячейка означает отсутствие наблюдения.
9,OpenSupply,PO_ID,Текст,Обезличенный идентификатор отдельной учебной поставки.


## Шаг 4. Условия задачи

Читаем лист "Условия" — явные допущения и параметры, заданные заказчиком (срок поставки, календарь и т.п.).

In [15]:
conditions_df = pd.read_excel(dictionary_path, sheet_name="Условия")
conditions_df

,Параметр,Значение
0,Дата расчёта,"19.09.2025, до размещения заказа"
1,История,19.06.2024–18.09.2025; 457 дней; 200 SKU
2,Гранулярность,SKU × календарный день
3,Конец дня,Столбец «Остаток на конец дня» относится к дню Date.
4,Единицы,Все количества — штуки; внутренние перемещения возможны поштучно.
5,Отсутствующие значения,Пустой остаток — наблюдения нет. Все SKU активного ассортимента остаются в итоговом расчёте.
6,История продаж,Фактически выполненные продажи. Исторического журнала неудовлетворённого спроса нет.
7,Учебные условия,"LeadTimeDays, календарь заказов и строки OpenSupply заданы для задания; это не реальные параметры компании."
8,Поставки,InTransit и OpenPO — разные состояния. Сводные количества SKU и строки OpenSupply описывают один набор поставок.
9,Исторические поставки,OpenSupply — снимок только на дату расчёта. Для backtesting требуется явно задать начальное состояние и правила моделирования.


## Шаг 5. Лист SKU

Загружаем таблицу активного ассортимента: текущие остатки, открытые поставки, срок поставки.

In [16]:
# Лист SKU — одна строка на позицию активного ассортимента (ожидаем 200 строк).
sku_df = pd.read_excel(candidate_path, sheet_name="SKU")

print("Форма таблицы (строки, колонки):", sku_df.shape)
print("Колонки:", list(sku_df.columns))
sku_df.head()

Форма таблицы (строки, колонки): (200, 5)
Колонки: ['SKU', 'CurrentStock', 'InTransit', 'OpenPO', 'LeadTimeDays']


,SKU,CurrentStock,InTransit,OpenPO,LeadTimeDays
0,TEST-0001,0,0,0,10
1,TEST-0002,3,0,1,10
2,TEST-0003,19,4,7,10
3,TEST-0004,0,0,0,10
4,TEST-0005,18,2,0,10


## Шаг 6. Лист DailyHistory

Загружаем дневную историю продаж и остатков — самая большая таблица (200 SKU × 457 дней).

In [17]:
history_df = pd.read_excel(candidate_path, sheet_name="DailyHistory")

print("Форма таблицы (строки, колонки):", history_df.shape)
print("Колонки:", list(history_df.columns))
print("Диапазон дат:", history_df["Date"].min(), "—", history_df["Date"].max())
history_df.head()

Форма таблицы (строки, колонки): (91400, 4)
Колонки: ['Date', 'SKU', 'SalesQty', 'Остаток на конец дня']
Диапазон дат: 2024-06-19 00:00:00 — 2025-09-18 00:00:00


,Date,SKU,SalesQty,Остаток на конец дня
0,2024-06-19,TEST-0001,0,2.0
1,2024-06-20,TEST-0001,0,2.0
2,2024-06-21,TEST-0001,0,2.0
3,2024-06-22,TEST-0001,0,2.0
4,2024-06-23,TEST-0001,0,2.0


## Шаг 7. Лист OpenSupply

Загружаем список уже размещённых поставок (InTransit/OpenPO) — снимок на дату расчёта.

In [18]:
open_supply_df = pd.read_excel(candidate_path, sheet_name="OpenSupply")

print("Форма таблицы (строки, колонки):", open_supply_df.shape)
print("Колонки:", list(open_supply_df.columns))
open_supply_df.head()

Форма таблицы (строки, колонки): (199, 6)
Колонки: ['PO_ID', 'SKU', 'Qty', 'Status', 'PlacedDate', 'ExpectedReceiptDate']


,PO_ID,SKU,Qty,Status,PlacedDate,ExpectedReceiptDate
0,PO-0002-1,TEST-0002,1,OpenPO,2025-09-15,2025-09-24
1,PO-0003-1,TEST-0003,4,InTransit,2025-09-14,2025-09-27
2,PO-0003-2,TEST-0003,7,OpenPO,2025-09-13,2025-10-04
3,PO-0005-1,TEST-0005,2,InTransit,2025-09-12,2025-10-03
4,PO-0006-1,TEST-0006,1,OpenPO,2025-09-11,2025-10-06


## Шаг 8. Лист OrderCalendar

Загружаем календарь размещения заказов — понадобится для расчёта protection period.

In [19]:
order_calendar_df = pd.read_excel(candidate_path, sheet_name="OrderCalendar")

print("Форма таблицы (строки, колонки):", order_calendar_df.shape)
print("Колонки:", list(order_calendar_df.columns))
order_calendar_df.head()

Форма таблицы (строки, колонки): (157, 3)
Колонки: ['OrderDate', 'NextOrderDate', 'OrderCycleDays']


,OrderDate,NextOrderDate,OrderCycleDays
0,2024-06-19,2024-06-21,2
1,2024-06-21,2024-06-26,5
2,2024-06-26,2024-06-28,2
3,2024-06-28,2024-07-03,5
4,2024-07-03,2024-07-05,2


## Шаг 9. Полнота календарной истории

Проверяем, что в `DailyHistory` ровно одна строка на каждую пару SKU-день, без пропущенных дней и без дублей.

In [20]:
# Считаем число строк, число уникальных пар SKU+Date и ожидаемое число (SKU x дни).
n_rows = len(history_df)
n_unique_pairs = history_df[["SKU", "Date"]].drop_duplicates().shape[0]
n_expected = sku_df["SKU"].nunique() * history_df["Date"].nunique()

print("Строк в DailyHistory:", n_rows)
print("Уникальных пар SKU+Date:", n_unique_pairs)
print("Ожидается (SKU x дни):", n_expected)
print("Дубликатов пар SKU+Date:", n_rows - n_unique_pairs)

Строк в DailyHistory: 91400
Уникальных пар SKU+Date: 91400
Ожидается (SKU x дни): 91400
Дубликатов пар SKU+Date: 0


## Шаг 10. Пропуски в данных

Считаем долю пропусков в `SalesQty` и в «Остаток на конец дня» (по словарю полей — пустой остаток значит «нет наблюдения», а не ноль).

In [21]:
missing_sales = history_df["SalesQty"].isna().sum()
missing_stock = history_df["Остаток на конец дня"].isna().sum()

print(f"Пропуски в SalesQty: {missing_sales} ({missing_sales / n_rows:.2%})")
print(f"Пропуски в 'Остаток на конец дня': {missing_stock} ({missing_stock / n_rows:.2%})")

Пропуски в SalesQty: 0 (0.00%)
Пропуски в 'Остаток на конец дня': 1489 (1.63%)


## Шаг 11. Активность продаж по SKU

Для каждого SKU считаем долю дней с продажами и суммарный спрос — чтобы понять, насколько распространён прерывистый спрос и есть ли SKU совсем без истории продаж.

In [22]:
# Группируем по SKU и считаем базовые показатели активности спроса.
sku_activity = history_df.groupby("SKU").agg(
    days_total=("SalesQty", "size"),
    days_with_sales=("SalesQty", lambda s: (s > 0).sum()),
    total_sales=("SalesQty", "sum"),
    mean_daily_sales=("SalesQty", "mean"),
).reset_index()

# Доля дней с продажами — ключевой индикатор прерывистости спроса.
sku_activity["share_days_with_sales"] = sku_activity["days_with_sales"] / sku_activity["days_total"]

print("SKU без единой продажи за всю историю:", (sku_activity["total_sales"] == 0).sum())
print("Медианная доля дней с продажами по SKU:", round(sku_activity["share_days_with_sales"].median(), 4))
print("SKU с долей дней с продажами < 10%:", (sku_activity["share_days_with_sales"] < 0.10).sum())

sku_activity.sort_values("total_sales").head(10)

SKU без единой продажи за всю историю: 18
Медианная доля дней с продажами по SKU: 0.0098
SKU с долей дней с продажами < 10%: 170


,SKU,days_total,days_with_sales,total_sales,mean_daily_sales,share_days_with_sales
7,TEST-0008,457,0,0,0.0,0.0
11,TEST-0012,457,0,0,0.0,0.0
29,TEST-0030,457,0,0,0.0,0.0
18,TEST-0019,457,0,0,0.0,0.0
49,TEST-0050,457,0,0,0.0,0.0
59,TEST-0060,457,0,0,0.0,0.0
43,TEST-0044,457,0,0,0.0,0.0
34,TEST-0035,457,0,0,0.0,0.0
60,TEST-0061,457,0,0,0.0,0.0
77,TEST-0078,457,0,0,0.0,0.0


## Шаг 12. Проверка на очевидные аномалии

Ищем отрицательные значения продаж/остатков и другие явно некорректные данные.

In [23]:
print("Отрицательные значения SalesQty:", (history_df["SalesQty"] < 0).sum())
print("Отрицательные значения остатка:", (history_df["Остаток на конец дня"] < 0).sum())
print("Максимальная продажа за день:", history_df["SalesQty"].max())
print("Максимальный остаток на конец дня:", history_df["Остаток на конец дня"].max())

Отрицательные значения SalesQty: 0
Отрицательные значения остатка: 0
Максимальная продажа за день: 150
Максимальный остаток на конец дня: 559.0


### Выводы и наблюдения

- Календарь наблюдений полный: 91 400 строк, дублей нет, пропущенных дней нет.
- `SalesQty` заполнен без единого пропуска — для наивной оценки спроса дополнительная очистка не требуется.
- В «Остаток на конец дня» — 1,63% пропусков (нет наблюдения, не ноль). Для baseline не критично (спрос оцениваем по `SalesQty`), но важно для бэктеста в разделе 2, где понадобится стартовое состояние запасов.
- Спрос по большинству SKU сильно прерывистый: медианная доля дней с продажами — **0,98%**, у 170 из 200 SKU (85%) доля дней с продажами меньше 10%. Наивное среднее по всей истории это переживёт, но будет давать грубую оценку — это ожидаемая точка для доработки в разделе 2, а не проблема, которую нужно решать прямо сейчас.
- **18 SKU (9%) не имеют вообще ни одной продажи за все 457 дней.** Для них наивная оценка спроса даст 0 — понадобится явный fallback и пометка в итоговой таблице (по условию задания такие позиции нельзя просто пропустить).
- Отрицательных значений и явных выбросов не найдено (максимум продаж за день — 150 шт., максимум остатка — 559 шт.) — данные в этой части чистые.

**Отдельный шаг «приведение данных в порядок» пропускаем** — для baseline он оказался бы пустым: `SalesQty` без пропусков/дублей/отрицательных значений, а два найденных момента (пропуски остатка, SKU без истории) — это не грязные данные, а два случая, которые нужно явно обработать логикой модели, а не почистить заранее.

## Шаг 13. Protection period — интервал до следующего заказа

Дата расчёта — это дата размещения заказа. Заказанное сегодня количество должно закрыть спрос не до следующей даты заказа, а до прихода *следующей за ней* поставки — поэтому берём интервал до следующего заказа из `OrderCalendar`, не хардкодим.

In [24]:
# Находим в календаре строку, где OrderDate — это наша дата расчёта.
calc_date_row = order_calendar_df[order_calendar_df["OrderDate"] == CALCULATION_DATE]

# Ожидаем ровно одну строку — на каждую разрешённую дату заказа календарь даёт один интервал до следующей.
assert len(calc_date_row) == 1, f"Ожидали одну строку календаря на {CALCULATION_DATE.date()}, нашли {len(calc_date_row)}"

# OrderCycleDays — сколько дней пройдёт до следующей возможности заказать (2 после среды, 5 после пятницы).
order_cycle_days = int(calc_date_row["OrderCycleDays"].iloc[0])
next_order_date = calc_date_row["NextOrderDate"].iloc[0]

print("Дата расчёта:", CALCULATION_DATE.date())
print("Следующая дата заказа:", next_order_date.date())
print("Интервал до следующего заказа (OrderCycleDays):", order_cycle_days, "дней")

Дата расчёта: 2025-09-19
Следующая дата заказа: 2025-09-24
Интервал до следующего заказа (OrderCycleDays): 5 дней


## Шаг 14. Protection period по каждому SKU

`LeadTimeDays` — колонка в `SKU`, а не глобальная константа, поэтому считаем protection period на уровне SKU (хотя по условию задания срок поставки для всех одинаковый, код не должен зависеть от этого совпадения).

In [25]:
# Начинаем сборку рабочей таблицы baseline_df с копии SKU, чтобы не менять исходный sku_df.
baseline_df = sku_df.copy()

# Protection period = срок поставки нового заказа (LeadTimeDays) + интервал до следующего заказа.
baseline_df["protection_period_days"] = baseline_df["LeadTimeDays"] + order_cycle_days

print("Уникальные значения LeadTimeDays:", sorted(baseline_df["LeadTimeDays"].unique()))
print("Уникальные значения protection_period_days:", sorted(baseline_df["protection_period_days"].unique()))

Уникальные значения LeadTimeDays: [np.int64(10)]
Уникальные значения protection_period_days: [np.int64(15)]


## Шаг 15. Наивная оценка спроса

Берём среднесуточный спрос по всей истории (уже посчитан в `sku_activity` на шаге 11 как `mean_daily_sales` — среднее по ВСЕМ дням, включая дни без продаж, что и есть корректная оценка интенсивности спроса). Умножаем на protection period.

In [26]:
# Подтягиваем среднесуточный спрос по SKU из таблицы активности (шаг 11).
baseline_df = baseline_df.merge(
    sku_activity[["SKU", "mean_daily_sales"]],
    on="SKU",
    how="left",
)

# Ожидаемый спрос за protection period — простое произведение, без поправок на дефицит.
baseline_df["demand_over_protection"] = baseline_df["mean_daily_sales"] * baseline_df["protection_period_days"]

print("SKU с нулевым ожидаемым спросом за protection period:", (baseline_df["demand_over_protection"] == 0).sum())
baseline_df[["SKU", "mean_daily_sales", "protection_period_days", "demand_over_protection"]].describe()

SKU с нулевым ожидаемым спросом за protection period: 18


,mean_daily_sales,protection_period_days,demand_over_protection
count,200.000000,200.0,200.000000
mean,0.374333,15.0,5.614989
std,1.131424,0.0,16.971359
min,0.000000,15.0,0.000000
25%,0.006565,15.0,0.098468
50%,0.031729,15.0,0.475930
75%,0.234683,15.0,3.520241
max,10.557987,15.0,158.369803


## Шаг 16. Order-up-to level (S) и Inventory Position — baseline

Baseline сознательно простой — без страхового запаса и без фильтрации поставок по горизонту прихода (это не забыто, а осознанно отложено: baseline должен быть "наивной" точкой сравнения для более аккуратной модели в разделе 2, а не черновиком финальной модели).

- `S = demand_over_protection` (без safety stock).
- `InventoryPosition = CurrentStock + InTransit + OpenPO` (без фильтра по дате прихода).

In [27]:
# Order-up-to level для baseline — без страхового запаса, чистое ожидание спроса.
baseline_df["S_baseline"] = baseline_df["demand_over_protection"]

# Позиция запаса — текущий остаток плюс всё, что уже в пути или заказано.
baseline_df["inventory_position"] = (
    baseline_df["CurrentStock"] + baseline_df["InTransit"] + baseline_df["OpenPO"]
)

baseline_df[["SKU", "S_baseline", "inventory_position"]].describe()

,S_baseline,inventory_position
count,200.000000,200.000000
mean,5.614989,15.515000
std,16.971359,33.771621
min,0.000000,0.000000
25%,0.098468,2.000000
50%,0.475930,4.000000
75%,3.520241,13.000000
max,158.369803,236.000000


## Шаг 17. RecommendedOrder — baseline

`RecommendedOrder = max(0, round(S - InventoryPosition))` — целое, неотрицательное, как требует задание.

In [28]:
# round() до целого, затем max(0, ...) — неотрицательное количество.
gap = baseline_df["S_baseline"] - baseline_df["inventory_position"]
baseline_df["RecommendedOrder_baseline"] = gap.round().clip(lower=0).astype(int)

print("Строк в baseline_df:", len(baseline_df))
print("Все 200 SKU на месте:", set(baseline_df["SKU"]) == set(sku_df["SKU"]))
print("Суммарный объём заказа по baseline:", baseline_df["RecommendedOrder_baseline"].sum(), "шт.")
print("SKU с RecommendedOrder > 0:", (baseline_df["RecommendedOrder_baseline"] > 0).sum())

baseline_df[["SKU", "S_baseline", "inventory_position", "RecommendedOrder_baseline"]].sort_values(
    "RecommendedOrder_baseline", ascending=False
).head(10)

Строк в baseline_df: 200
Все 200 SKU на месте: True
Суммарный объём заказа по baseline: 97 шт.
SKU с RecommendedOrder > 0: 18


,SKU,S_baseline,inventory_position,RecommendedOrder_baseline
135,TEST-0136,22.910284,0,23
24,TEST-0025,96.695842,85,12
48,TEST-0049,9.518600,0,10
33,TEST-0034,7.483589,0,7
81,TEST-0082,15.689278,9,7
108,TEST-0109,6.630197,0,7
51,TEST-0052,7.385120,0,7
129,TEST-0130,7.582057,3,5
162,TEST-0163,4.135667,0,4
90,TEST-0091,3.971554,0,4


### Выводы и наблюдения

- Protection period на 19.09.2025 = 10 (лид-тайм) + 5 (интервал до следующего заказа, из календаря) = **15 дней**, одинаков для всех SKU (`LeadTimeDays` в данных везде равен 10 — но код это не предполагает заранее, а вычисляет).
- Baseline-формула прогнана на всех 200 SKU без ошибок, `RecommendedOrder` — целое и неотрицательное у всех строк.
- Суммарный объём заказа по baseline — **97 шт.**, но только у **18 из 200 SKU** (9%) получилось `RecommendedOrder > 0`. Это ожидаемо, а не баг: без страхового запаса заказ появляется только там, где текущей позиции запаса не хватает даже на голый средний спрос за 15 дней — для позиций с прерывистым спросом это редкий случай.
- 18 SKU без истории продаж (найдены в EDA) получили `RecommendedOrder_baseline = 0` — это корректное поведение формулы (нулевой спрос → нулевая рекомендация), но по условию задания для таких позиций нужна явная пометка в итоговой таблице. Формулу это не меняет, оставляем на раздел 3 (там же, где собирается финальная таблица).
- Baseline осознанно **не включает**: страховой запас, поправку на дефицит (censored demand), фильтрацию поставок по горизонту прихода. Это не недоработка, а точка сравнения — та же формула (`S_baseline`, `inventory_position`) переиспользуется без изменений как простой baseline при сравнении с доработанной моделью в разделе 2.

# Раздел 2. Бэктесты и отладка

## Шаг 18. Выбор контрольных дат

Берём даты заказа из `OrderCalendar` (кроме самой даты расчёта 19.09.2025 — она зарезервирована под финальный расчёт в разделе 3). Валидна дата, если после неё в истории есть данные минимум на весь её protection period вперёд — иначе нам не с чем будет сравнить рекомендацию (не увидим реальный спрос за этот период). Из валидных берём 8 последних — не нужно перебирать все 156, для проверки модели достаточно.

In [29]:
# На шаге 14 подтвердили, что LeadTimeDays одинаков у всех SKU — фиксируем это явной проверкой
# (а не молчаливым предположением) и используем как скаляр для расчётов на уровне календаря.
assert sku_df["LeadTimeDays"].nunique() == 1, "LeadTimeDays неоднороден - protection period нельзя считать на уровне календаря одним числом"
LEAD_TIME_DAYS = int(sku_df["LeadTimeDays"].iloc[0])

In [30]:
# Последний доступный день фактической истории — дальше него сравнивать не с чем.
last_history_date = history_df["Date"].max()

# protection period одинаков для всех SKU (LeadTimeDays константа), поэтому считаем его на уровне календаря,
# а не на уровне SKU, как в baseline — тут это одно число на дату, а не на строку.
calendar_check = order_calendar_df.copy()
calendar_check["protection_period_days"] = calendar_check["OrderCycleDays"] + LEAD_TIME_DAYS
calendar_check["last_covered_date"] = calendar_check["OrderDate"] + pd.to_timedelta(
    calendar_check["protection_period_days"], unit="D"
)

# Валидные даты: не сама дата расчёта, и после них в истории есть данные на весь protection period.
is_not_calc_date = calendar_check["OrderDate"] != CALCULATION_DATE
has_enough_future_history = calendar_check["last_covered_date"] <= last_history_date
valid_control_dates = calendar_check[is_not_calc_date & has_enough_future_history]

# Берём 8 последних валидных дат — они ближе всего по времени к дате расчёта.
N_CONTROL_DATES = 8
control_dates_df = valid_control_dates.sort_values("OrderDate").tail(N_CONTROL_DATES).reset_index(drop=True)

print("Всего дат в календаре:", len(order_calendar_df))
print("Валидных дат (хватает будущей истории):", len(valid_control_dates))
print("Выбрано контрольных дат:", len(control_dates_df))
control_dates_df[["OrderDate", "OrderCycleDays", "protection_period_days", "last_covered_date"]]

Всего дат в календаре: 157
Валидных дат (хватает будущей истории): 127
Выбрано контрольных дат: 8


,OrderDate,OrderCycleDays,protection_period_days,last_covered_date
0,2025-08-08,5,15,2025-08-23
1,2025-08-13,2,12,2025-08-25
2,2025-08-15,5,15,2025-08-30
3,2025-08-20,2,12,2025-09-01
4,2025-08-22,5,15,2025-09-06
5,2025-08-27,2,12,2025-09-08
6,2025-08-29,5,15,2025-09-13
7,2025-09-03,2,12,2025-09-15


## Шаг 18.1. Разделение на подбор параметров и проверку

Чтобы не подбирать Service Level и не проверять итоговый результат на одних и тех же данных, делим 8 контрольных дат по времени: более ранние — для подбора параметра (чувствительность, шаг 31), более поздние (ближе к дате расчёта) — для итоговой проверки/сравнения с baseline (шаг 30). Это не идеальное разделение (дат мало), но оно даёт честную проверку вместо подбора и оценки на одном и том же наборе.

In [31]:
# Более ранние 5 дат - подбор параметра (TUNE), более поздние 3 - независимая проверка (TEST).
tune_dates_df = control_dates_df.iloc[:5].reset_index(drop=True)
test_dates_df = control_dates_df.iloc[5:].reset_index(drop=True)

print("TUNE (подбор SL):", tune_dates_df["OrderDate"].dt.date.tolist())
print("TEST (проверка/сравнение с baseline):", test_dates_df["OrderDate"].dt.date.tolist())

TUNE (подбор SL): [datetime.date(2025, 8, 8), datetime.date(2025, 8, 13), datetime.date(2025, 8, 15), datetime.date(2025, 8, 20), datetime.date(2025, 8, 22)]
TEST (проверка/сравнение с baseline): [datetime.date(2025, 8, 27), datetime.date(2025, 8, 29), datetime.date(2025, 9, 3)]


## Шаг 19. Допущения бэктеста

Исторического журнала заказов и открытых поставок нет — снимок `OpenSupply` есть только на 19.09.2025. Поэтому для каждой контрольной даты явно задаём:

1. **Стартовая позиция запаса** = фактический остаток на конец дня, предшествующего контрольной дате (`Остаток на конец дня` за `T − 1`), из реальной истории. `InTransit`/`OpenPO` на исторические даты принимаем за 0 — у нас нет данных, что реально было в пути в прошлом, а не потому что их не было.
2. **Оценка спроса** считается только по истории **строго до** контрольной даты — без утечки будущего в параметры модели.
3. **Проверка решения — по сумме за весь protection period, без разбиения на "до/после прихода заказа".** `shortfall = max(0, факт_спрос_за_период − (inventory_position + RecommendedOrder))`. Осознанно не делим период на фазы: решение о размере заказа принимается один раз, до начала периода, и никак не может повлиять на то, случится ли дефицит именно в первые `LeadTimeDays` дней, пока заказ ещё в пути — сколько бы мы ни заказали, это не изменится. Разбиение на фазы не изменило бы наше решение сегодня, только усложнило бы проверку. **Ограничение, которое это даёт:** метрика может переоценивать реальный сервис у SKU, чей спрос сильно смещён к началу периода (до прихода заказа) — дефицит там реален, но по сумме за весь период может "спрятаться" за профицитом во второй половине. Это отмечено как известное ограничение, а не скрытая ошибка.

## Шаг 20. Остаток с заполнением пропусков (для подстановки на любую дату)

На EDA нашли 1,63% пропусков в «Остаток на конец дня» — для итогового расчёта (раздел 3) это было не нужно, но для бэктеста нужно уметь взять остаток на *любую* историческую дату, в т.ч. там, где наблюдения нет. Заполняем пропуски последним известным значением по каждому SKU (вперёд по времени) — самое простое и защитимое допущение: "остаток не наблюдали, но и явных признаков, что он изменился, тоже нет".

In [32]:
# Сортируем по SKU и дате — обязательное условие для корректного forward-fill внутри каждого SKU.
history_sorted = history_df.sort_values(["SKU", "Date"]).copy()

# ffill внутри каждой группы SKU: пропуск заполняется последним известным остатком этого же SKU.
history_sorted["stock_filled"] = history_sorted.groupby("SKU")["Остаток на конец дня"].ffill()

# Сколько осталось пропусков после ffill — должны остаться только самые первые дни истории SKU,
# если у него вообще нет ни одного наблюдения остатка до какой-то точки.
remaining_missing = history_sorted["stock_filled"].isna().sum()
print("Пропусков после ffill:", remaining_missing, f"({remaining_missing / len(history_sorted):.3%})")

Пропусков после ffill: 1093 (1.196%)


## Шаг 21. Проверка: protection period на произвольную дату

Функция `get_protection_period_days` (раздел "Функции") обобщает логику шага 13 — проверяем, что она даёт тот же результат.

In [33]:
# Проверяем функцию (определена в разделе "Функции") на уже известном результате шага 13 (19.09.2025 -> 15 дней).
assert get_protection_period_days(CALCULATION_DATE) == 15
print("Функция проверена на дате расчёта: 15 дней — совпадает с шагом 13.")

Функция проверена на дате расчёта: 15 дней — совпадает с шагом 13.


## Шаг 22. Проверка: наивная оценка спроса по данным до даты

Функция `get_naive_demand_estimate` (раздел "Функции") — ключевое отличие от baseline в том, что здесь считаем только по истории, доступной до конкретной даты решения (иначе утечёт будущее).

In [34]:
# Проверяем функцию (определена в разделе "Функции") на дате расчёта: там использовали среднее по ВСЕЙ
# истории (в ней и так нет дней после 18.09.2025, так что результат должен совпасть).
check = get_naive_demand_estimate(CALCULATION_DATE)
comparison = sku_activity.set_index("SKU")["mean_daily_sales"].compare(check.reindex(sku_activity["SKU"]))
print("Расхождений с шагом 15 (полная история):", len(comparison))

Расхождений с шагом 15 (полная история): 0


## Шаг 23. Проверка: позиция запаса на историческую дату

Функция `get_inventory_position_asof` (раздел "Функции") — по допущению шага 19: фактический остаток на конец предыдущего дня, `InTransit`/`OpenPO` = 0 (для исторических дат этих данных нет).

In [35]:
# Проверяем функцию (определена в разделе "Функции") на первой контрольной дате — сколько SKU
# потребовали fallback на 0.
test_position, test_missing = get_inventory_position_asof(control_dates_df["OrderDate"].iloc[0])
print("SKU с известной позицией запаса:", (test_position.notna()).sum() - test_missing, "из 200")
print("SKU, для которых позицию запаса пришлось принять за 0 (нет ни одного наблюдения):", test_missing)

SKU с известной позицией запаса: 198 из 200
SKU, для которых позицию запаса пришлось принять за 0 (нет ни одного наблюдения): 2


## Шаг 24. Фактический спрос за protection period

Для каждой (SKU, дата) считаем реальные продажи с `control_date` по `control_date + protection_period_days − 1` включительно — это то, с чем будем сравнивать решение модели. Считаем сразу по всем 8 датам (TUNE + TEST) — понадобится и baseline, и доработанной модели, поэтому вынесено перед обоими расчётами.

In [36]:
# Считаем фактический спрос по всем 8 контрольным датам (TUNE + TEST) разом,
# функцией get_actual_demand_over_window из раздела "Функции".
actual_demand_parts = []
for _, cal_row in control_dates_df.iterrows():
    control_date = cal_row["OrderDate"]
    protection_period_days = int(cal_row["protection_period_days"])
    part = get_actual_demand_over_window(control_date, protection_period_days).reset_index()
    part["control_date"] = control_date
    actual_demand_parts.append(part)

actual_demand_df = pd.concat(actual_demand_parts, ignore_index=True)

print("Строк в actual_demand_df:", len(actual_demand_df))
actual_demand_df.head()

Строк в actual_demand_df: 1600


,SKU,actual_demand,control_date
0,TEST-0001,0,2025-08-08
1,TEST-0002,2,2025-08-08
2,TEST-0003,5,2025-08-08
3,TEST-0004,0,2025-08-08
4,TEST-0005,5,2025-08-08


## Шаг 25. Решения baseline-модели на TEST-датах

Baseline не имеет настраиваемых параметров, но для честного сравнения с доработанной моделью (шаг 30) считаем его на тех же TEST-датах, что и итоговую проверку — не на всех 8 сразу.

In [37]:
# Итоговое сравнение baseline считаем на TEST-датах - тех же, что и для доработанной модели (шаг 29),
# функцией compute_baseline_decisions из раздела "Функции".
baseline_decisions_df = compute_baseline_decisions(test_dates_df)

print("Строк (SKU x TEST-даты):", len(baseline_decisions_df))
print("Пропуски в ключевых колонках:", baseline_decisions_df[["mean_daily_sales", "inventory_position", "RecommendedOrder"]].isna().sum().to_dict())
baseline_decisions_df.head()

Строк (SKU x TEST-даты): 600


Пропуски в ключевых колонках: {'mean_daily_sales': 0, 'inventory_position': 0, 'RecommendedOrder': 0}


,SKU,control_date,protection_period_days,mean_daily_sales,inventory_position,S,RecommendedOrder,actual_demand,available_to_cover,shortfall,excess,cycle_success
0,TEST-0001,2025-08-27,12,0.004608,0.0,0.055300,0,0,0.0,0.0,0.0,True
1,TEST-0002,2025-08-27,12,0.004608,3.0,0.055300,0,0,3.0,0.0,3.0,True
2,TEST-0003,2025-08-27,12,0.366359,28.0,4.396313,0,7,28.0,0.0,21.0,True
3,TEST-0004,2025-08-27,12,0.027650,0.0,0.331797,0,0,0.0,0.0,0.0,True
4,TEST-0005,2025-08-27,12,0.223502,16.0,2.682028,0,2,16.0,0.0,14.0,True


## Шаг 26. Дефицит, излишек, успешность цикла

Как договорились: без разбиения на фазы, по сумме за весь period.

In [38]:
# Сколько всего будет доступно за period: то, что уже есть, плюс то, что закажем.
baseline_decisions_df["available_to_cover"] = (
    baseline_decisions_df["inventory_position"] + baseline_decisions_df["RecommendedOrder"]
)

# Дефицит — если факт спроса превысил доступное; излишек — если наоборот.
baseline_decisions_df["shortfall"] = (
    baseline_decisions_df["actual_demand"] - baseline_decisions_df["available_to_cover"]
).clip(lower=0)
baseline_decisions_df["excess"] = (
    baseline_decisions_df["available_to_cover"] - baseline_decisions_df["actual_demand"]
).clip(lower=0)

# Цикл успешен, если дефицита не было вообще.
baseline_decisions_df["cycle_success"] = baseline_decisions_df["shortfall"] == 0

baseline_decisions_df[["SKU", "control_date", "available_to_cover", "actual_demand", "shortfall", "excess", "cycle_success"]].head()

,SKU,control_date,available_to_cover,actual_demand,shortfall,excess,cycle_success
0,TEST-0001,2025-08-27,0.0,0,0.0,0.0,True
1,TEST-0002,2025-08-27,3.0,0,0.0,3.0,True
2,TEST-0003,2025-08-27,28.0,7,0.0,21.0,True
3,TEST-0004,2025-08-27,0.0,0,0.0,0.0,True
4,TEST-0005,2025-08-27,16.0,2,0.0,14.0,True


### Определения метрик

- **Cycle Service Level** — доля циклов (пар SKU × дата) без единого дня дефицита за весь protection period. Бинарная метрика: не важно, не хватило на 1 шт. или на 100 — цикл либо "успешен", либо нет. `= mean(shortfall == 0)`.
- **Fill Rate** — доля от общего объёма спроса (в штуках), которая была бы покрыта. Непрерывная метрика, взвешенная по объёму, а не по числу решений. `= 1 - Σ(shortfall) / Σ(actual_demand)`.
- **Средний / максимальный запас** — среднее и максимум `available_to_cover` (`inventory_position + RecommendedOrder`) по всем циклам — сколько товара физически предполагалось иметь на цикл.
- **Дефицит (shortfall)** и **излишек (excess)** — соответственно `max(0, факт - available_to_cover)` и `max(0, available_to_cover - факт)`.

**Что измеримо напрямую, а что — допущение:** факт продаж (`SalesQty`) и факт остатка на дату наблюдения — прямые измерения из данных. Стартовая позиция запаса на исторические даты (без `InTransit`/`OpenPO`) и сама механика "проверка по сумме за period, без разбиения на фазы" (шаг 19) — допущения, введённые из-за отсутствия исторического журнала поставок и заказов, а не измеренные величины.

## Шаг 27. Метрики baseline на TEST-датах

Cycle Service Level, Fill Rate, средний/максимальный запас, суммарный дефицит/излишек — определения см. выше.

In [39]:
# compute_metrics определена в разделе "Функции".
baseline_metrics_df = compute_metrics(baseline_decisions_df, "Baseline")
baseline_metrics_df

,model,cycle_service_level,fill_rate,avg_inventory,max_inventory,total_shortfall,total_excess
0,Baseline,0.951667,0.806305,13.085,219.0,639.0,5191.0


### Выводы и наблюдения

- Baseline на 3 TEST-датах (600 решений SKU×дата): **Cycle Service Level 95,2%**, но **Fill Rate только 80,6%**.
- Разрыв между метриками — не ошибка, а ожидаемое следствие сильно прерывистого спроса: у большинства (SKU, дата) фактический спрос почти нулевой, поэтому "не уйти в дефицит" — тривиально легко, отсюда высокий Cycle Service Level. А редкие решения с реально заметным спросом промахиваются на существенный объём (нет страхового запаса) — и именно они утягивают вниз Fill Rate, который взвешен по штукам, а не по количеству решений.
- Средний запас — 13,1 шт., но максимум доходит до 219 — сильная неоднородность между "спокойными" и "объёмными" SKU.
- Суммарный излишек (5 191 шт.) более чем в 8 раз больше суммарного дефицита (639 шт.) — типичная картина для baseline без страхового запаса на лампи-спросе: у "тихих" SKU почти любой остаток избыточен относительно их мизерного среднего спроса, а у "объёмных" SKU среднее как оценка систематически недооценивает реальные всплески.

Это прямая мотивация для доработки: следующий шаг — заменить чистое среднее на "дни защиты + страховой запас" с учётом изменчивости спроса, чтобы целенаправленно закрыть недостающие 19,4% спроса в штуках, не раздувая и без того избыточный запас у стабильных SKU.

## Шаг 28. Доработанная модель: дни защиты + страховой запас

Первая попытка (эмпирический квантиль всей суммы спроса за period) давала хорошие метрики сервиса, но раздувала запас почти вдвое — тест показал, что цена улучшения слишком высокая. Возвращаемся к классической и более управляемой структуре из двух явных слагаемых:

```
S = мат. ожидание спроса за protection period + страховой запас
```

где страховой запас = `z(Service Level) × σ(спрос за period)` — но не по сырым продажам, а по продажам **с поправкой на дефицит** (censored demand): выше в задании отдельно указано учесть "наличие и отсутствие товара" — а мы это ещё не сделали.

**Поправка на дефицит:** 23,5% SKU-дней имеют наблюдаемый (не пропущенный) нулевой остаток на конец дня. Средние продажи в такие дни — 0,11 шт., против 0,46 шт. в дни с положительным остатком (в 4 раза меньше) — характерный признак censored demand: как только товара не было, продажи могли занижать реальный спрос. Для таких дней заменяем `SalesQty` на среднее по этому же SKU в дни, когда товар был в наличии (тот же принцип, что и для пропусков остатка на шаге 20, только для другого типа "пропуска" — не отсутствия наблюдения, а отсутствия товара).

In [40]:
# Проверяем функцию (раздел "Функции") на первой TEST-дате - сколько дней и на сколько скорректировалось.
test_corrected = get_corrected_past_history(test_dates_df["OrderDate"].iloc[0])
n_corrected = (test_corrected["SalesQty_corrected"] != test_corrected["SalesQty"]).sum()
print("Скорректированных SKU-дней:", n_corrected, f"({n_corrected / len(test_corrected):.2%})")
print("Средняя поправка (было -> стало) по скорректированным дням:")
print(test_corrected.loc[test_corrected["SalesQty_corrected"] != test_corrected["SalesQty"], ["SalesQty", "SalesQty_corrected"]].mean())

Скорректированных SKU-дней: 10890 (12.55%)
Средняя поправка (было -> стало) по скорректированным дням:


SalesQty              0.215611
SalesQty_corrected    0.414353
dtype: float64


In [41]:
# Проверяем функцию (раздел "Функции") на первой TEST-дате.
test_S = get_S_with_safety_stock(test_dates_df["OrderDate"].iloc[0], 15, SERVICE_LEVEL)
print("z-квантиль для SL=95%:", round(norm.ppf(SERVICE_LEVEL), 3))
test_S.describe()

z-квантиль для SL=95%: 1.645


count    200.000000
mean      13.415554
std       35.127126
min        0.000000
25%        0.371704
50%        1.663686
75%        8.534747
max      315.467160
Name: SalesQty_corrected, dtype: float64

## Шаг 29. Решения доработанной модели на TEST-датах

Та же механика, что и для baseline (шаг 24), на тех же TEST-датах — `S` теперь из `get_S_with_safety_stock` (дни защиты + страховой запас, с поправкой на дефицит), а не из квантиля или простого среднего.

In [42]:
# Итоговое сравнение считаем на TEST-датах - не участвовавших в подборе Service Level (шаг 31),
# функцией build_decisions из раздела "Функции".
improved_decisions_df = build_decisions(test_dates_df, get_S_with_safety_stock, service_level=SERVICE_LEVEL)

print("Строк:", len(improved_decisions_df))
print("Пропуски:", improved_decisions_df[["S", "inventory_position", "actual_demand"]].isna().sum().to_dict())
improved_decisions_df.head()

Строк: 600
Пропуски: {'S': 0, 'inventory_position': 0, 'actual_demand': 0}


,SKU,control_date,protection_period_days,S,inventory_position,RecommendedOrder,actual_demand,available_to_cover,shortfall,excess,cycle_success
0,TEST-0001,2025-08-27,12,0.000000,0.0,0,0,0.0,0.0,0.0,True
1,TEST-0002,2025-08-27,12,0.411262,3.0,0,0,3.0,0.0,3.0,True
2,TEST-0003,2025-08-27,12,12.127327,28.0,0,7,28.0,0.0,21.0,True
3,TEST-0004,2025-08-27,12,2.171201,0.0,2,0,2.0,0.0,2.0,True
4,TEST-0005,2025-08-27,12,6.256011,16.0,0,2,16.0,0.0,14.0,True


## Шаг 30. Сравнение с baseline

Считаем метрики доработанной модели той же функцией `compute_metrics` (шаг 27) и кладём рядом с baseline в одну таблицу.

In [43]:
improved_metrics_df = compute_metrics(improved_decisions_df, f"Доработанная модель (SL={SERVICE_LEVEL:.0%})")

comparison_df = pd.concat([baseline_metrics_df, improved_metrics_df], ignore_index=True)
comparison_df

,model,cycle_service_level,fill_rate,avg_inventory,max_inventory,total_shortfall,total_excess
0,Baseline,0.951667,0.806305,13.085000,219.0,639.0,5191.0
1,Доработанная модель (SL=95%),0.973333,0.896029,16.898333,315.0,343.0,7183.0


### Выводы и наблюдения

- Доработанная модель (дни защиты + страховой запас, с поправкой на дефицит, SL=95%) на TEST-датах против baseline: **Cycle Service Level 95,2% → 97,3%**, **Fill Rate 80,6% → 89,6%** — заметное улучшение по обеим метрикам.
- Средний запас вырос с 13,1 до 16,9 шт. (+29%) — умеренная цена, значительно ниже, чем при первой попытке через сырой эмпирический квантиль (там рост был почти двукратным при сопоставимом сервисе) — явная причина, почему формула была изменена на "дни защиты + отдельный страховой запас".
- Суммарный дефицит упал почти вдвое (639 → 343 шт.).
- Метрики посчитаны на 3 TEST-датах (600 решений) — сознательно отделены от 5 TUNE-дат, на которых подбирался Service Level (шаг 31), чтобы не оценивать модель на тех же данных, по которым её настраивали.

## Шаг 31. Чувствительность к уровню сервиса (на TUNE-датах)

Пересчитываем доработанную модель при Service Level 90/95/98% — **на TUNE-датах** (шаг 18.1), не на тех, по которым отчитываемся в сравнении с baseline (шаг 30). Так выбор параметра не подсматривает в данные, на которых потом же его и хвалим.

In [44]:
# Baseline на TUNE-датах - точка отсчёта для сравнения чувствительности (не итоговое сравнение).
tune_baseline_decisions = compute_baseline_decisions(tune_dates_df)
sensitivity_rows = [compute_metrics(tune_baseline_decisions, "Baseline (TUNE)")]

# Прогоняем доработанную модель при трёх уровнях сервиса - переиспользуем build_decisions.
for sl in [0.90, 0.95, 0.98]:
    decisions = build_decisions(tune_dates_df, get_S_with_safety_stock, service_level=sl)
    metrics = compute_metrics(decisions, f"SL={sl:.0%} (TUNE)")
    sensitivity_rows.append(metrics)

sensitivity_df = pd.concat(sensitivity_rows, ignore_index=True)
sensitivity_df

,model,cycle_service_level,fill_rate,avg_inventory,max_inventory,total_shortfall,total_excess
0,Baseline (TUNE),0.950,0.879326,15.005,326.0,652.0,10254.0
1,SL=90% (TUNE),0.968,0.935591,17.571,326.0,348.0,12516.0
2,SL=95% (TUNE),0.974,0.946696,18.355,326.0,288.0,13240.0
3,SL=98% (TUNE),0.980,0.956321,19.434,354.0,236.0,14267.0


### Выводы по разделу 2

**Чувствительность (на TUNE-датах):** рост Service Level с 90% до 98% даёт монотонный компромисс — Fill Rate растёт с 93,6% до 95,6%, средний запас растёт с 17,6 до 19,4 шт.

**Выбор итогового уровня сервиса — 95%.** Задание не фиксирует целевой Service Level числом явно. Отдача от роста запаса на единицу Fill Rate немного замедляется после 95% (с 90% до 95% — +1,1 п.п. Fill Rate на +0,8 шт. запаса; с 95% до 98% — уже +0,9 п.п. на +1,0 шт.) — 95% выбран как разумная точка баланса, не крайняя по обе стороны диапазона.

**Разделение периодов (шаг 18.1):** 5 более ранних дат (TUNE) использованы для подбора Service Level, 3 более поздние (TEST) — для итогового сравнения с baseline (шаг 30), чтобы не оценивать модель на тех же данных, на которых она настраивалась.

**Ограничения методологии бэктеста (честно, не скрываем):**
- Проверка решения — по сумме за весь период, без учёта, что заказ физически появляется только после lead time (шаг 19) — может переоценивать сервис у SKU со спросом, смещённым к началу периода. Осознанный выбор: размер сегодняшнего заказа всё равно не может повлиять на дефицит в первые `LeadTimeDays` дней, поэтому разбиение на фазы не изменило бы само решение.
- `InTransit`/`OpenPO` = 0 для всех исторических контрольных дат — исторического журнала поставок нет физически. Метрики бэктеста, вероятно, консервативны. (Для финального расчёта на 19.09.2025, где реальные данные о поставках есть, это не допущение, а точный учёт — раздел 3.)
- Всего 8 контрольных дат, из них только 3 - в итоговом TEST-сравнении — небольшая выборка, метрики раздела 2 стоит воспринимать как направление, а не как точную величину с узким доверительным интервалом.

Аномалии и ограничения самих данных (пропуски, SKU без истории и т.п.) зафиксированы в разделе 1 (EDA) — здесь не дублируем.

# Раздел 3. Заказ

## Шаг 32. Финальный расчёт на 19.09.2025

Один разовый прогон доработанной модели (раздел 2, дни защиты + страховой запас, Service Level 95%) на реальную дату расчёта. Два отличия от бэктеста:
1. Настоящая позиция запаса из `SKU` (`CurrentStock`), а не историческое приближение.
2. **`InTransit`/`OpenPO` учитываются только если `ExpectedReceiptDate` попадает в protection period** — поставка, которая придёт позже, физически не может закрыть спрос в этом цикле, поэтому засчитывать её сейчас было бы некорректно.

In [45]:
# protection period на дату расчёта - тот же самый, что уже проверен на шаге 13 (15 дней).
final_protection_period_days = get_protection_period_days(CALCULATION_DATE)

# S по доработанной модели - дни защиты + страховой запас (шаг 28), на реальной дате расчёта.
final_S = get_S_with_safety_stock(CALCULATION_DATE, final_protection_period_days, SERVICE_LEVEL)

# Горизонт, до которого поставка ещё успевает помочь в этом цикле.
horizon_end = CALCULATION_DATE + pd.Timedelta(days=final_protection_period_days - 1)

# Из всех открытых поставок учитываем только те, что прибудут не позже горизонта.
supply_in_horizon = open_supply_df[open_supply_df["ExpectedReceiptDate"] <= horizon_end]
supply_out_of_horizon = open_supply_df[open_supply_df["ExpectedReceiptDate"] > horizon_end]
supply_in_horizon_by_sku = supply_in_horizon.groupby("SKU")["Qty"].sum()

# Позиция запаса = реальный текущий остаток + только те поставки, что попадают в горизонт.
final_inventory_position = (
    sku_df.set_index("SKU")["CurrentStock"] + supply_in_horizon_by_sku.reindex(sku_df["SKU"]).fillna(0.0)
).rename("inventory_position")

print("Protection period:", final_protection_period_days, "дней, горизонт до", horizon_end.date())
print("Поставок всего:", len(open_supply_df))
print("Поставок в горизонте (учтены):", len(supply_in_horizon))
print("Поставок позже горизонта (не учтены):", len(supply_out_of_horizon), f"({len(supply_out_of_horizon) / len(open_supply_df):.1%})")
final_S.describe()

Protection period: 15 дней, горизонт до 2025-10-03
Поставок всего: 199
Поставок в горизонте (учтены): 140
Поставок позже горизонта (не учтены): 59 (29.6%)


count    200.000000
mean      13.453928
std       34.927709
min        0.000000
25%        0.465401
50%        1.568484
75%        8.597688
max      309.626788
Name: SalesQty_corrected, dtype: float64

## Шаг 33. RecommendedOrder + пометка для SKU без истории продаж

По условию задания позиции без достаточной истории нельзя просто пропустить — нужно явное решение с комментарием. У нас это 18 SKU, у которых за все 457 дней не было ни одной продажи (найдены в EDA, шаг 11) — модель для них честно возвращает 0 (нет данных, чтобы предположить другое), а не потому что спрос низкий, но измеримый.

In [46]:
final_order_df = pd.DataFrame({"SKU": sku_df["SKU"]})
final_order_df = final_order_df.merge(final_S.rename("S"), on="SKU", how="left")
final_order_df = final_order_df.merge(final_inventory_position, on="SKU", how="left")

# Формула не меняется - та же, что и везде в проекте.
gap = final_order_df["S"] - final_order_df["inventory_position"]
final_order_df["RecommendedOrder"] = gap.round().clip(lower=0).astype(int)

# SKU без единой продажи за всю историю (шаг 11) - явная пометка, а не молчаливый ноль.
zero_history_skus = set(sku_activity.loc[sku_activity["total_sales"] == 0, "SKU"])
final_order_df["Комментарий"] = final_order_df["SKU"].apply(
    lambda sku: (
        "Нет истории продаж за 457 дней - рекомендация 0 отражает отсутствие данных, "
        "а не подтверждённый нулевой спрос. Требует ручной проверки перед заказом."
    ) if sku in zero_history_skus else ""
)

print("Строк:", len(final_order_df))
print("Все 200 SKU на месте:", set(final_order_df["SKU"]) == set(sku_df["SKU"]))
print("RecommendedOrder целые и неотрицательные:", (final_order_df["RecommendedOrder"] >= 0).all() and final_order_df["RecommendedOrder"].dtype.kind == "i")
print("SKU с пометкой 'нет истории':", (final_order_df["Комментарий"] != "").sum())
print("Суммарный объём заказа:", final_order_df["RecommendedOrder"].sum(), "шт.")

final_order_df.sort_values("RecommendedOrder", ascending=False).head(10)

Строк: 200
Все 200 SKU на месте: True
RecommendedOrder целые и неотрицательные: True
SKU с пометкой 'нет истории': 18
Суммарный объём заказа: 799 шт.


,SKU,S,inventory_position,RecommendedOrder,Комментарий
24,TEST-0025,178.438523,85.0,93,
42,TEST-0043,309.626788,236.0,74,
117,TEST-0118,95.048135,26.0,69,
135,TEST-0136,57.896506,0.0,58,
48,TEST-0049,47.386300,0.0,47,
51,TEST-0052,35.173580,0.0,35,
68,TEST-0069,46.699072,12.0,35,
81,TEST-0082,40.929967,9.0,32,
181,TEST-0182,60.820761,31.0,30,
164,TEST-0165,124.641156,97.0,28,


## Шаг 34. Экспорт в RecommendedOrder.xlsx

In [47]:
# Папка output/ уже существует в проекте (с .gitkeep) - собираем путь относительно корня.
output_dir = project_dir / "output"
output_path = output_dir / "RecommendedOrder.xlsx"

# Экспортируем только итоговые колонки, которые нужны по условию задания.
final_order_df[["SKU", "RecommendedOrder", "Комментарий"]].to_excel(output_path, index=False)

print("Сохранено:", output_path)

# Перечитываем сохранённый файл - проверка, что экспорт действительно корректен, а не просто "не упал".
check_df = pd.read_excel(output_path)
print("Строк в сохранённом файле:", len(check_df))
print("Колонки:", list(check_df.columns))

Сохранено: C:\Users\User\Desktop\clode folder\projects_code\тестовое от Генерации\reorder-model\output\RecommendedOrder.xlsx
Строк в сохранённом файле: 200
Колонки: ['SKU', 'RecommendedOrder', 'Комментарий']


### Выводы и итоги проекта

**Финальная рекомендация на 19.09.2025:** 200 SKU, суммарно **799 шт.** к заказу.

**Методика в двух словах:**
- Спрос оценивается как "дни защиты + страховой запас": среднесуточный спрос (с поправкой на дефицит) × protection period, плюс `z(Service Level) × эмпирическое std` скользящих сумм спроса за такие периоды.
- **Поправка на дефицит**: в 23,5% SKU-дней остаток на конец дня наблюдался нулевым, и продажи в такие дни в среднем в 4 раза ниже — признак censored demand. Такие дни заменяются на среднее по SKU в дни с товаром в наличии (шаг 28).
- Protection period = срок поставки (10 дней) + интервал до следующего заказа из `OrderCalendar`, вычисляется, а не хардкодится.
- **Поставки (`InTransit`/`OpenPO`) учитываются только если успевают прийти в защитный период** — из 199 открытых поставок 59 (29,6%) прибывают позже горизонта и в расчёт не идут (шаг 32).
- Service Level = 95%, выбран по чувствительности (раздел 2, шаг 31) на данных, не участвовавших в итоговой проверке (шаг 18.1).
- Модель проверена бэктестом на 3 независимых TEST-датах, отдельных от 5 TUNE-дат подбора параметра.
- 18 SKU без истории продаж — явно помечены в итоговой таблице, а не скрыты под нулём.

**Результаты бэктеста (TEST-даты):** доработанная модель против baseline — **Cycle Service Level 95,2% → 97,3%**, **Fill Rate 80,6% → 89,6%**, ценой роста среднего запаса на 29% (13,1 → 16,9 шт.).

**Основные ограничения (полный список — в разделе 2):**
- Бэктест не учитывает точный момент прихода заказа внутри protection period (сумма за весь период, а не по фазам) — осознанный выбор, не влияющий на само решение о размере заказа.
- Историю поставок (`InTransit`/`OpenPO`) для бэктеста пришлось принять за 0 — исторического журнала нет физически. Для финального расчёта это не проблема — там используются реальные данные `OpenSupply`.
- Всего 8 контрольных дат (3 из них — итоговый TEST-набор) — небольшая выборка, метрики раздела 2 стоит воспринимать как направление, а не точную величину.
- 2 SKU (TEST-0040, TEST-0169) имеют длинные провалы в наблюдениях остатка — на финальный расчёт это не влияет (используется текущий снимок `SKU`), но ограничивает точность их участия в бэктесте.

**Направления развития** (если понадобится больше времени): явные модели интермиттентного спроса (Croston/SBA) вместо эмпирической коррекции, посуточная симуляция бэктеста вместо суммы за период, учёт стоимости дефицита/хранения для экономически обоснованного выбора Service Level, расширение TUNE/TEST выборки контрольных дат.